# Pembagian Dataset (Stratified Split 80:20) & Pembobotan TF-IDF (Unigram + Bigram)

Notebook ini mengimplementasikan **Subbab 3.8 (Pembobotan TF-IDF)** dan **Subbab 3.9 (Pembagian Dataset)** pada proposal skripsi.

### Alur Proses pada Notebook Ini:
1. **Membaca Data Hasil Preprocessing** (`data_step4_preprocessed.csv`).
2. **Memisahkan Fitur ($X$) dan Target Label ($y$)**.
3. **Pembagian Dataset (80% Data Latih : 20% Data Uji)** menggunakan teknik **Stratified Split** agar proporsi kelas Positif dan Negatif tetap seimbang.
4. **Pembobotan Kata TF-IDF dengan Kombinasi Unigram dan Bigram** (`ngram_range=(1, 2)`):
   - `fit_transform()` pada **Data Latih** (membentuk *vocabulary* & menghitung bobot IDF).
   - `transform()` pada **Data Uji** (mencegah *data leakage* / kebocoran informasi dari data uji).
5. **Eksplorasi & Bedah Hasil Perhitungan TF-IDF** (untuk bahan tabel & pembahasan di **Bab IV Skripsi**).
6. **Menyimpan Hasil Split & Matriks TF-IDF** agar siap digunakan oleh algoritma **Multinomial Naive Bayes** dan **K-Nearest Neighbor (KNN)** pada kondisi pengujian yang 100% setara.

---
## 1. Import Library & Membaca Data Hasil Text Preprocessing

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Membaca dataset hasil text preprocessing (tahap 4)
file_input = "../data/data_step4_preprocessed.csv"
df = pd.read_csv(file_input)

# Pastikan tidak ada baris kosong (NaN) pada kolom 'text_clean'
df = df.dropna(subset=['text_clean']).copy()
df['text_clean'] = df['text_clean'].astype(str)
df = df[df['text_clean'].str.strip() != ''].reset_index(drop=True)

print(f"Total data siap proses: {len(df)} ulasan")
print("\nDistribusi Kelas Sentimen:")
distribusi = df['sentiment'].value_counts()
persentase = df['sentiment'].value_counts(normalize=True) * 100
for label in distribusi.index:
    print(f"- {label.capitalize():8s}: {distribusi[label]} ulasan ({persentase[label]:.2f}%)")

df[['content', 'text_clean', 'sentiment']].head()

Total data siap proses: 10732 ulasan

Distribusi Kelas Sentimen:
- Positif : 5640 ulasan (52.55%)
- Negatif : 5092 ulasan (47.45%)


,content,text_clean,sentiment
0,makasii,makasii,positif
1,mudah digubakan dan bermanfaat,mudah guna manfaat,positif
2,aplikasi tidak mendukung yg os 13&12,aplikasi tidak dukung os,negatif
3,sata ingin memesan gogood tapi malah ga ada op...,mes gofood malah tidak opsi antar hemat jadi d...,negatif
4,"aplikasi gajelas, kebanyakan di cancel driver,...",aplikasi gajelas banyak cancel kemudi sering d...,negatif


---
## 2. Menentukan Variabel Fitur ($X$) dan Target Label ($y$)

Dalam pembelajaran mesin terawasi (*Supervised Learning*), kita membagi kolom dataset menjadi dua komponen:
- **Fitur ($X$ / Variabel Independen):** Data masukan yang akan dipelajari polanya oleh model, yaitu kolom `text_clean` (teks ulasan yang sudah melalui 6 tahap *preprocessing*).
- **Target ($y$ / Variabel Dependen / Label):** Kelas tujuan yang ingin diprediksi oleh model, yaitu kolom `sentiment` (`positif` atau `negatif`).

In [2]:
# Memisahkan fitur teks (X) dan label sentimen (y)
X = df['text_clean']
y = df['sentiment']

print(f"Jumlah data Fitur (X) : {len(X)}")
print(f"Jumlah data Target (y): {len(y)}")

Jumlah data Fitur (X) : 10732
Jumlah data Target (y): 10732


---
## 3. Pembagian Dataset dengan Stratified Split (80% Latih : 20% Uji)

**Sesuai Subbab 3.9 Proposal Skripsi:**
- Dataset dibagi menjadi **80% Data Latih (*Training Data*)** dan **20% Data Uji (*Testing Data*)**.
- **Mengapa dibagi sebelum TF-IDF?** Agar model dan perhitungan bobot kata (IDF) murni belajar dari **Data Latih** saja, sehingga tidak terjadi kebocoran informasi (*Data Leakage*) dari Data Uji.

**Penjelasan Parameter `train_test_split`:**
1. `test_size=0.2` : Menetapkan 20% data untuk pengujian (*testing*) dan 80% sisanya untuk pelatihan (*training*).
2. `stratify=y` (**Stratified Split**) : Menjaga proporsi kelas `positif` dan `negatif` pada Data Latih dan Data Uji agar tetap sama persis dengan proporsi pada dataset utuh.
3. `random_state=42` : Mengunci angka acak (*seed*) agar setiap kali kode ini dijalankan ulang, urutan data yang masuk ke Data Latih dan Data Uji tidak berubah-ubah (*reproducible*).

In [9]:
# Membagi dataset 80% Data Latih dan 20% Data Uji dengan Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Membuat tabel ringkasan pembagian dataset (Siap dimasukkan ke Bab IV Skripsi)
ringkasan_split = pd.DataFrame({
    'Dataset': ['Data Latih (Training 80%)', 'Data Uji (Testing 20%)', 'Total Keseluruhan'],
    'Jumlah Data': [len(X_train), len(X_test), len(X)],
    'Positif': [
        (y_train == 'positif').sum(),
        (y_test == 'positif').sum(),
        (y == 'positif').sum()
    ],
    'Negatif': [
        (y_train == 'negatif').sum(),
        (y_test == 'negatif').sum(),
        (y == 'negatif').sum()
    ],
    '% Positif': [
        f"{(y_train == 'positif').mean() * 100:.2f}%",
        f"{(y_test == 'positif').mean() * 100:.2f}%",
        f"{(y == 'positif').mean() * 100:.2f}%"
    ],
    '% Negatif': [
        f"{(y_train == 'negatif').mean() * 100:.2f}%",
        f"{(y_test == 'negatif').mean() * 100:.2f}%",
        f"{(y == 'negatif').mean() * 100:.2f}%"
    ]
})

print("Tabel Ringkasan Pembagian Dataset (Stratified Split 80:20):")
ringkasan_split

Tabel Ringkasan Pembagian Dataset (Stratified Split 80:20):


,Dataset,Jumlah Data,Positif,Negatif,% Positif,% Negatif
0,Data Latih (Training 80%),8585,4512,4073,52.56%,47.44%
1,Data Uji (Testing 20%),2147,1128,1019,52.54%,47.46%
2,Total Keseluruhan,10732,5640,5092,52.55%,47.45%


---
## 4. Pembobotan Kata TF-IDF dengan Kombinasi Unigram dan Bigram

**Sesuai Subbab 3.8 Proposal Skripsi:**
Algoritma *Machine Learning* (Multinomial Naive Bayes & KNN) tidak dapat menghitung teks secara langsung, sehingga teks harus diubah menjadi matriks angka menggunakan metode **TF-IDF (*Term Frequency - Inverse Document Frequency*)**.

### Konsep Dasar yang Perlu Dipahami:
1. **Unigram & Bigram (`ngram_range=(1, 2)`):**
   - **Unigram (1 kata):** Mengambil setiap kata tunggal sebagai fitur, misal: `"aplikasi"`, `"tidak"`, `"bagus"`.
   - **Bigram (2 kata berurutan):** Mengambil pasangan 2 kata yang berdampingan sebagai fitur, misal: `"aplikasi tidak"`, `"tidak bagus"`.
   - *Kenapa Bigram penting?* Agar pasangan kata negasi seperti `"tidak bagus"` atau `"kurang cepat"` tertangkap sebagai satu kesatuan makna!
2. **Term Frequency ($TF$):** Menghitung seberapa sering suatu kata/frasa muncul di dalam satu ulasan.
3. **Inverse Document Frequency ($IDF$):** Mengukur seberapa unik/penting suatu kata di seluruh ulasan. Kata yang muncul di hampir semua ulasan (misal kata `"aplikasi"` atau `"gojek"`) akan mendapat bobot IDF kecil, sedangkan kata spesifik penentu sentimen akan mendapat bobot IDF lebih besar.
4. **Perbedaan `fit_transform()` vs `transform()` (Pencegahan *Data Leakage*):**
   - Pada **Data Latih (`X_train`)**, kita gunakan `.fit_transform(X_train)` untuk mempelajari kamus kata (*vocabulary*) dan menghitung nilai IDF dari data latih, lalu mengubahnya menjadi matriks bobot TF-IDF.
   - Pada **Data Uji (`X_test`)**, kita **HANYA** menggunakan `.transform(X_test)` agar data uji dinilai menggunakan kamus kata dan bobot IDF yang sudah dipelajari dari data latih.

In [10]:
# Inisialisasi TfidfVectorizer dengan kombinasi Unigram (1 kata) dan Bigram (2 kata)
# ngram_range=(1, 2) berarti minimal 1 kata (unigram) dan maksimal 2 kata (bigram)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))

# 1. fit_transform pada Data Latih (mempelajari vocabulary + IDF, sekaligus mengubah ke matriks TF-IDF)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# 2. transform pada Data Uji (mengubah data uji ke matriks TF-IDF berdasarkan vocabulary Data Latih)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Mengambil daftar seluruh fitur (kata/frasa) yang terbentuk dari Data Latih
fitur_names = tfidf_vectorizer.get_feature_names_out()

# Memisahkan fitur Unigram (tidak mengandung spasi) dan Bigram (mengandung spasi)
fitur_unigram = [f for f in fitur_names if ' ' not in f]
fitur_bigram = [f for f in fitur_names if ' ' in f]

print("=== HASIL PEMBENTUKAN FITUR TF-IDF (UNIGRAM + BIGRAM) ===")
print(f"Ukuran Matriks TF-IDF Data Latih : {X_train_tfidf.shape[0]} dokumen x {X_train_tfidf.shape[1]} fitur")
print(f"Ukuran Matriks TF-IDF Data Uji   : {X_test_tfidf.shape[0]} dokumen x {X_test_tfidf.shape[1]} fitur")
print("-" * 58)
print(f"Total Fitur (Vocabulary) Terbentuk : {len(fitur_names):,} fitur")
print(f"  -> Jumlah Fitur Unigram (1 kata) : {len(fitur_unigram):,} fitur")
print(f"  -> Jumlah Fitur Bigram  (2 kata) : {len(fitur_bigram):,} fitur")
print("-" * 58)
print(f"Contoh 10 Fitur Unigram : {fitur_unigram[:10]}")
print(f"Contoh 10 Fitur Bigram  : {fitur_bigram[:10]}")

=== HASIL PEMBENTUKAN FITUR TF-IDF (UNIGRAM + BIGRAM) ===
Ukuran Matriks TF-IDF Data Latih : 8585 dokumen x 57901 fitur
Ukuran Matriks TF-IDF Data Uji   : 2147 dokumen x 57901 fitur
----------------------------------------------------------
Total Fitur (Vocabulary) Terbentuk : 57,901 fitur
  -> Jumlah Fitur Unigram (1 kata) : 7,524 fitur
  -> Jumlah Fitur Bigram  (2 kata) : 50,377 fitur
----------------------------------------------------------
Contoh 10 Fitur Unigram : ['aa', 'aada', 'aagak', 'aala', 'aamiin', 'aaminnn', 'aaya', 'aba', 'abai', 'abales']
Contoh 10 Fitur Bigram  : ['aa gojek', 'aada problem', 'aagak lebih', 'aala nyata', 'aamiin pesan', 'aamiin rabbal', 'aamiin thanks', 'aaya kasi', 'aba mau', 'abai gamkom']


---
## 5. Bedah Hasil Perhitungan TF-IDF pada Sampel Ulasan (Untuk Bahan Bab IV Skripsi)

Saat menyusun **Bab IV (Hasil dan Pembahasan)**, dosen penguji atau pembimbing biasanya meminta Anda menampilkan **contoh nyata bagaimana satu ulasan diubah menjadi fitur Unigram, Bigram, nilai IDF, dan bobot TF-IDF-nya**.

Kode di bawah ini mengambil 1 sampel ulasan dari Data Latih dan menampilkan rincian bobot setiap fitur (Unigram & Bigram) yang terkandung di dalam ulasan tersebut.

In [11]:
# Ambil 1 contoh dokumen dari Data Latih (misalnya baris ke-0 pada X_train)
indeks_sampel = 0
teks_sampel = X_train.iloc[indeks_sampel]
label_sampel = y_train.iloc[indeks_sampel]

print(f"Teks Ulasan (Setelah Preprocessing) : '{teks_sampel}'")
print(f"Label Sentimen                      : {label_sampel}")

# Ambil vektor TF-IDF untuk dokumen sampel tersebut
vektor_sampel = X_train_tfidf[indeks_sampel]

# Ambil indeks fitur yang nilainya tidak nol (fitur yang muncul di dalam ulasan sampel ini)
indeks_fitur_aktif = vektor_sampel.nonzero()[1]

# Susun ke dalam tabel DataFrame agar mudah dibaca
data_detail_tfidf = []
for idx in indeks_fitur_aktif:
    nama_fitur = fitur_names[idx]
    jenis_fitur = "Bigram (2 Kata)" if " " in nama_fitur else "Unigram (1 Kata)"
    nilai_idf = tfidf_vectorizer.idf_[idx]
    nilai_tfidf = vektor_sampel[0, idx]
    data_detail_tfidf.append({
        'Fitur (Term)': nama_fitur,
        'Jenis Fitur': jenis_fitur,
        'Nilai IDF': round(nilai_idf, 4),
        'Bobot TF-IDF': round(nilai_tfidf, 4)
    })

df_contoh_tfidf = pd.DataFrame(data_detail_tfidf).sort_values(by='Bobot TF-IDF', ascending=False).reset_index(drop=True)
print("\nTabel Rincian Fitur & Bobot TF-IDF pada Ulasan Sampel:")
df_contoh_tfidf

Teks Ulasan (Setelah Preprocessing) : 'sekarang gojek mahal bangett kadang dapet kemudi kurang sopan kadang suka marah marah kalau kasih tau jalan malah kadang suka ngeyel bagaimana pelang butuh cepet tidak macett'
Label Sentimen                      : negatif

Tabel Rincian Fitur & Bobot TF-IDF pada Ulasan Sampel:


,Fitur (Term),Jenis Fitur,Nilai IDF,Bobot TF-IDF
0,kadang,Unigram (1 Kata),5.2457,0.2849
1,kadang suka,Bigram (2 Kata),7.8607,0.2846
2,marah,Unigram (1 Kata),6.4744,0.2344
3,ngeyel bagaimana,Bigram (2 Kata),9.3647,0.1695
4,kadang dapet,Bigram (2 Kata),9.3647,0.1695
5,bangett kadang,Bigram (2 Kata),9.3647,0.1695
6,pelang butuh,Bigram (2 Kata),9.3647,0.1695
7,tidak macett,Bigram (2 Kata),9.3647,0.1695
8,mahal bangett,Bigram (2 Kata),9.3647,0.1695
9,bagaimana pelang,Bigram (2 Kata),9.3647,0.1695


### Melihat 15 Fitur (Unigram & Bigram) dengan Rata-Rata Bobot TF-IDF Tertinggi di Data Latih
Bagian ini membantu kita melihat kata atau frasa apa saja yang paling dominan di dalam korpus ulasan Gojek.

In [12]:
# Menghitung rata-rata bobot TF-IDF setiap fitur di seluruh dokumen Data Latih
rata_rata_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()

df_top_fitur = pd.DataFrame({
    'Fitur (Term)': fitur_names,
    'Jenis': ['Bigram' if ' ' in f else 'Unigram' for f in fitur_names],
    'Nilai IDF': tfidf_vectorizer.idf_,
    'Rata-rata TF-IDF': rata_rata_tfidf
}).sort_values(by='Rata-rata TF-IDF', ascending=False).reset_index(drop=True)

print("Top 15 Fitur dengan Rata-rata Bobot TF-IDF Tertinggi:")
df_top_fitur.head(15)

Top 15 Fitur dengan Rata-rata Bobot TF-IDF Tertinggi:


,Fitur (Term),Jenis,Nilai IDF,Rata-rata TF-IDF
0,sangat,Unigram,2.582549,0.026212
1,tidak,Unigram,2.208955,0.025528
2,kemudi,Unigram,2.387926,0.024242
3,aplikasi,Unigram,2.603168,0.020995
4,bagus,Unigram,3.558101,0.018133
5,gojek,Unigram,2.756740,0.017510
6,bantu,Unigram,3.610582,0.015181
7,baik,Unigram,3.515416,0.013818
8,layan,Unigram,3.698314,0.011336
9,cepat,Unigram,3.939791,0.011014


---
## 6. Menyimpan Hasil Pembagian Data & Matriks TF-IDF

**Kenapa perlu disimpan?**
Sesuai metodologi penelitian, metode **Multinomial Naive Bayes** dan **K-Nearest Neighbor (KNN)** harus diuji menggunakan **Data Latih, Data Uji, dan Matriks TF-IDF yang sama persis** agar perbandingannya adil (*apple-to-apple*).

Oleh karena itu, kita menyimpan:
1. `../data/data_train_80.csv` & `../data/data_test_20.csv` : File CSV data latih dan data uji untuk dokumentasi.
2. `../data/tfidf_vectorizer.pkl` : Objek model TF-IDF yang sudah di-*fit* pada data latih.
3. `../data/tfidf_split_data.pkl` : Paket data siap pakai berisi `(X_train_tfidf, X_test_tfidf, y_train, y_test, X_train, X_test)` yang tinggal dimuat (*load*) di notebook pemodelan selanjutnya.

In [13]:
# 1. Simpan Data Latih dan Data Uji dalam bentuk CSV untuk dokumentasi
df_train_csv = pd.DataFrame({'text_clean': X_train, 'sentiment': y_train})
df_test_csv = pd.DataFrame({'text_clean': X_test, 'sentiment': y_test})

df_train_csv.to_csv("../data/data_train_80.csv", index=False)
df_test_csv.to_csv("../data/data_test_20.csv", index=False)

# 2. Simpan objek TfidfVectorizer
joblib.dump(tfidf_vectorizer, "../data/tfidf_vectorizer.pkl")

# 3. Simpan matriks TF-IDF beserta labelnya agar siap dipakai di notebook Naive Bayes & KNN
paket_data_tfidf = {
    'X_train_tfidf': X_train_tfidf,
    'X_test_tfidf': X_test_tfidf,
    'y_train': y_train,
    'y_test': y_test,
    'X_train_text': X_train,
    'X_test_text': X_test
}
joblib.dump(paket_data_tfidf, "../data/tfidf_split_data.pkl")

print("=== PENYIMPANAN BERHASIL ===")
print("1. Data Latih (80%) CSV     -> ../data/data_train_80.csv")
print("2. Data Uji   (20%) CSV     -> ../data/data_test_20.csv")
print("3. Model TF-IDF Vectorizer  -> ../data/tfidf_vectorizer.pkl")
print("4. Matriks TF-IDF & Split   -> ../data/tfidf_split_data.pkl")

=== PENYIMPANAN BERHASIL ===
1. Data Latih (80%) CSV     -> ../data/data_train_80.csv
2. Data Uji   (20%) CSV     -> ../data/data_test_20.csv
3. Model TF-IDF Vectorizer  -> ../data/tfidf_vectorizer.pkl
4. Matriks TF-IDF & Split   -> ../data/tfidf_split_data.pkl
